# Coding Homework 04: QR and the Four Fundamental Subspaces

This assignment continues the mini-transformer you began in Coding HW03. You will turn inner-product geometry into a QR algorithm, extract dot-product coordinates in an orthonormal basis, organize the four fundamental subspaces, and add reusable orthogonal weight initialization to `mini_transformer.py`.

**Estimated time:** 2–3 hours
**Submit:** this notebook and `mini_transformer.py`


## 0. Setup and cumulative-model check

Keep `mini_transformer.py` beside this notebook. Its HW03 attention functions are supplied as the completed starting point; your new HW04 work belongs in the same student-owned model file. No AI background is assumed: a **weight matrix** such as $W_Q$ changes each token vector into a query vector. In machine learning this is sometimes called a "projection," but it means a linear feature map; it need not be an orthogonal projection in the geometric sense. The function name `orthogonal_projection_matrices` is retained from the cumulative model API.


In [1]:
import numpy as np
import mini_transformer as mt

np.set_printoptions(precision=5, suppress=True)

X_check = np.array([[1., 0., 2.], [0., 1., 1.]])
I3 = np.eye(3)
assert mt.attention_head(X_check, I3, I3, I3).shape == (2, 3)
print("Setup complete; the cumulative HW03 attention head is available.")


Setup complete; the cumulative HW03 attention head is available.


## 1. Modified Gram–Schmidt

Suppose the linearly independent columns of $A$ are $a_1,\ldots,a_n$. Modified Gram–Schmidt builds orthonormal columns $q_j$ one at a time. For column $j$:

1. Start with $v=a_j$.
2. For each earlier $q_i$, compute $r_{ij}=q_i^T v$ and replace $v$ by $v-r_{ij}q_i$.
3. Set $r_{jj}=\lVert v\rVert$ and $q_j=v/r_{jj}$.

The word *modified* refers to taking each inner product with the current residual `v`. Your result should satisfy $A=QR$ and $Q^TQ=I$.


In [2]:
def modified_gram_schmidt(A, tol=1e-12):
    """Return the reduced QR factorization A = QR.

    The columns of A must be linearly independent.  Raise ValueError if a
    residual column is too small to normalize safely.
    """
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    for j in range(n):
        v = A[:, j].copy()  # Start with the j-th column of A

        # Orthogonalize v against previous q_i columns
        for i in range(j):
            r_ij = Q[:, i].T @ v
            R[i, j] = r_ij
            v = v - r_ij * Q[:, i]

        # Normalize v to get q_j
        r_jj = np.linalg.norm(v)
        if r_jj < tol:
            raise ValueError("Columns are not linearly independent or too close to being so.")
        R[j, j] = r_jj
        Q[:, j] = v / r_jj

    return Q, R


In [3]:
A = np.array([[1., 1.], [1., 0.], [0., 1.]])
Q, R = modified_gram_schmidt(A)
assert Q.shape == (3, 2) and R.shape == (2, 2)
assert np.allclose(Q.T @ Q, np.eye(2), atol=1e-10)
assert np.allclose(Q @ R, A, atol=1e-10)
assert np.allclose(R, np.triu(R))

try:
    modified_gram_schmidt(np.array([[1., 2.], [2., 4.]]))
except ValueError:
    pass
else:
    raise AssertionError("Dependent columns should raise ValueError")
print("Modified Gram--Schmidt checks passed.")


Modified Gram--Schmidt checks passed.


## 2. Dot-product coordinates from QR

The columns of $Q$ from reduced QR form an orthonormal basis for $\operatorname{Col}(A)$. For any input $b$, define its recorded coordinates by

$$c=Q^Tb.$$

Each entry is the dot product of $b$ with one column of $Q$. If $b=Qz$ lies in that column space, then $c=Q^TQz=z$: these are its exact basis coefficients. For a general $b$, the map may lose information. An input change $h$ with $Q^Th=0$ changes none of the recorded coordinates (Units 12–13).

Implement `orthonormal_coordinates(A, b)`: obtain `Q` using your modified Gram–Schmidt function, then return `Q.T @ b`, a one-dimensional array with one entry per column of `A`. The columns of `A` are assumed independent. This task requires only QR and dot products; closest-point theorems and projection-matrix properties are reserved for Unit 16.


In [4]:
def orthonormal_coordinates(A, b):
    """Return the dot-product coordinates Q.T @ b from reduced QR of A."""
    Q, _ = modified_gram_schmidt(A)
    return Q.T @ b


In [5]:
A_coords = np.array([[1., 0.], [1., 1.], [0., 1.]])
b = np.array([2., -1., 3.])
Q_coords, _ = modified_gram_schmidt(A_coords)
c = orthonormal_coordinates(A_coords, b)
assert c.shape == (2,)
assert np.allclose(c, [Q_coords[:, i] @ b for i in range(2)])

# Coordinates of a vector already expressed in this basis recover its coefficients.
z = np.array([2., -1.])
assert np.allclose(orthonormal_coordinates(A_coords, Q_coords @ z), z)

# A change perpendicular to both columns is invisible to these coordinates.
h = np.array([1., -1., 1.])
assert np.allclose(A_coords.T @ h, np.zeros(2))
assert np.allclose(orthonormal_coordinates(A_coords, b + h), c)
print("Orthonormal-coordinate checks passed.")


Orthonormal-coordinate checks passed.


## 3. The four fundamental subspaces

For $A\in\mathbb R^{m\times n}$ of rank $r$:

| Subspace | Lives in | Dimension |
|---|---|---:|
| column space $\operatorname{Col}(A)$ | $\mathbb R^m$ | $r$ |
| null space $\operatorname{Null}(A)$ | $\mathbb R^n$ | $n-r$ |
| row space $\operatorname{Row}(A)$ | $\mathbb R^n$ | $r$ |
| left null space $\operatorname{Null}(A^T)$ | $\mathbb R^m$ | $m-r$ |

Complete `subspace_dimensions`. Return a dictionary with keys `column_space`, `null_space`, `row_space`, and `left_null_space`.


In [6]:
def subspace_dimensions(A, tol=None):
    """Return the dimensions of A's four fundamental subspaces."""
    m, n = A.shape
    A_rank = np.linalg.matrix_rank(A, tol=tol)
    # Return a dictionary with the four keys shown in the prompt.
    return {
        "column_space": A_rank,
        "null_space": n - A_rank,
        "row_space": A_rank,
        "left_null_space": m - A_rank,
    }


In [7]:
A_four = np.array([[1., 2., 3., 4.],
                   [2., 4., 6., 8.],
                   [0., 1., 1., 0.]])
dims = subspace_dimensions(A_four)
assert dims == {
    "column_space": 2,
    "null_space": 2,
    "row_space": 2,
    "left_null_space": 1,
}
print("Four-subspace dimension checks passed:", dims)


Four-subspace dimension checks passed: {'column_space': np.int64(2), 'null_space': np.int64(2), 'row_space': np.int64(2), 'left_null_space': np.int64(1)}


## 4. Orthogonal initialization for the mini-transformer

An **initializer** chooses starting weight entries before training. A square orthogonal matrix $W$ satisfies $W^TW=WW^T=I$ and preserves lengths and dot products. QR supplies such matrices without choosing nearly parallel columns.

For a tall $W$ with orthonormal columns, $W^TW=I$ still holds. Consequently, for a coefficient column vector $z$, $\|Wz\|^2=z^TW^TWz=\|z\|^2$. Be careful about direction: mapping a row token $x$ to $xW$ can lose information when the feature count decreases. We do not require a general norm bound for that map here.

Open `mini_transformer.py` and implement:

- `orthogonal_matrix(rows, cols=None, seed=0)`, using a QR factorization of a random Gaussian matrix (entries drawn from a standard normal distribution). For a tall matrix require orthonormal columns; for a wide matrix construct a tall matrix and transpose it, giving orthonormal rows.
- `orthogonal_projection_matrices(d_model, d_head, seed=0)`, returning distinct `(W_Q, W_K, W_V)` weight matrices of shape `(d_model, d_head)`.

`seed` makes the initializer reproducible: the same seed should give the same matrices, while the three weight matrices should not be identical.


In [8]:
W_tall = mt.orthogonal_matrix(6, 3, seed=412)
W_wide = mt.orthogonal_matrix(3, 6, seed=412)
assert W_tall.shape == (6, 3)
assert W_wide.shape == (3, 6)
assert np.allclose(W_tall.T @ W_tall, np.eye(3), atol=1e-10)
assert np.allclose(W_wide @ W_wide.T, np.eye(3), atol=1e-10)

W_Q, W_K, W_V = mt.orthogonal_projection_matrices(6, 3, seed=412)
assert all(W.shape == (6, 3) for W in (W_Q, W_K, W_V))
assert not np.allclose(W_Q, W_K)
print("Orthogonal initializer checks passed.")


Orthogonal initializer checks passed.


## 5. Mini-transformer milestone: use the new weights

Each row of `X` is one token vector. The three orthogonally initialized weight matrices turn those token vectors into queries, keys, and values; the HW03 attention head then mixes the values according to query–key similarity. A causal mask prevents a token from using information from later positions.


In [9]:
X = np.array([
    [1., 0., 1., 0., 0., 1.],
    [0., 1., 1., 0., 1., 0.],
    [1., 1., 0., 1., 0., 0.],
    [0., 0., 1., 1., 1., 0.],
])
mask = mt.causal_mask(len(X))
W_Q, W_K, W_V = mt.orthogonal_projection_matrices(
    d_model=X.shape[1], d_head=3, seed=412
)
Y = mt.attention_head(X, W_Q, W_K, W_V, mask=mask)
assert Y.shape == (4, 3)
assert np.all(np.isfinite(Y))
print("Causal attention output:\n", Y)


Causal attention output:
 [[-1.33112  0.15714  0.3008 ]
 [-0.84261  0.14209  0.43025]
 [-0.85204  0.1553   0.59031]
 [-0.45878  0.30543  0.62281]]


### Short interpretation

In 2–4 sentences, use $W^TW=I$ to explain why $\|Wz\|=\|z\|$ for a coefficient column vector $z$. Does this imply that mapping a row token $x$ to $xW$ is one-to-one when `W` has more rows than columns? Explain using rank–nullity. No closest-point or projection theorem is needed.


In [10]:
orthogonal_initialization_explanation = """
If W^T W = I, then ||Wz||^2 = (Wz)^T (Wz) = z^T W^T W z = z^T z = ||z||^2, so ||Wz|| = ||z||.
This does not make the row map x |-> xW one-to-one when W is tall (more rows than columns).
That map goes from R^{d_model} to R^{d_head} with rank at most d_head < d_model, so rank-nullity gives a positive nullity:
some nonzero x satisfy xW = 0, and distinct tokens can collapse to the same features.
""".strip()

assert len(orthogonal_initialization_explanation.split()) >= 20
print(orthogonal_initialization_explanation)


If W^T W = I, then ||Wz||^2 = (Wz)^T (Wz) = z^T W^T W z = z^T z = ||z||^2, so ||Wz|| = ||z||.
This does not make the row map x |-> xW one-to-one when W is tall (more rows than columns).
That map goes from R^{d_model} to R^{d_head} with rank at most d_head < d_model, so rank-nullity gives a positive nullity:
some nonzero x satisfy xW = 0, and distinct tokens can collapse to the same features.


## 6. Submission and AI-use note

Restart the kernel, run all cells, and confirm every assertion passes. Keep this notebook alongside `mt_utils.py` / `mini_transformer.py` in the same folder when re-running.



In [11]:
ai_use_note = "I used AI to debug my orthogonal_matrix code function It was an issue with the if-else statement in returning Q."  # Replace if applicable
assert ai_use_note.strip()
print(ai_use_note)


I used AI to debug my orthogonal_matrix code function It was an issue with the if-else statement in returning Q.
